In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import tensorflow as tf
import pickle
from scipy.special import boxcox

from sklearn.model_selection import train_test_split, cross_val_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from keras import callbacks

In [13]:
def zscore_normalization(df, name):
    mean = df[name].mean()
    sd = df[name].std()

    df[name] = (df[name] - mean) / sd

def preprocess(df):
    df = df.drop(columns=['Name', 'md5'])
    for i in df.columns:
        if i != 'legitimate':
            df[i] = boxcox(df[i], 0.5)
            zscore_normalization(df, i)
    correlation_matrix = df.corr()
    cols_to_drop = []
    for i in df.columns:
        for j in df.columns:
            if i != j and i != 'legitimate' and j != 'legitimate' and abs(correlation_matrix[i][j]) > 0.6 and i not in cols_to_drop and j not in cols_to_drop:
                cols_to_drop.append(i)
    cols_to_drop = set(cols_to_drop)
    df.drop(columns=cols_to_drop, inplace=True)
    return df


In [29]:
df = pd.read_csv('MalwareData.csv', sep='|')
df = preprocess(df)

,SizeOfOptionalHeader,MinorLinkerVersion,SizeOfCode,SizeOfInitializedData,SizeOfUninitializedData,BaseOfCode,BaseOfData,ImageBase,SectionAlignment,MajorOperatingSystemVersion,...,ImportsNbOrdinal,ExportNb,ResourcesNb,ResourcesMinEntropy,ResourcesMaxEntropy,ResourcesMinSize,ResourcesMaxSize,LoadConfigurationSize,VersionInformationSize,legitimate
0,-0.360507,-0.376285,0.731982,-0.218904,-0.127541,-0.108158,0.844298,-0.014266,-0.027859,-3.783301,...,-0.386936,-0.246188,-0.364067,0.234965,-1.076480,0.020404,-0.077386,-0.027153,0.542462,1
1,-0.360507,-0.376285,0.052683,-0.617730,-0.127541,-0.108158,0.177571,-0.013153,-0.027859,0.092192,...,-0.386936,-0.246188,-0.536317,0.907994,-0.151389,0.080437,-0.292385,-0.014714,0.689509,1
2,-0.360507,-0.376285,1.063997,0.680065,-0.127541,-0.108158,1.678207,-0.013153,-0.027859,0.092192,...,1.972142,-0.034528,0.023085,0.465107,-0.046926,-0.013093,0.749509,-0.014714,0.689509,1
3,-0.360507,-0.376285,1.194639,0.317420,-0.127541,-0.108158,1.658516,-0.013153,-0.027859,0.092192,...,0.874044,-0.034528,-0.022301,0.319630,0.533257,-0.018389,-0.225282,-0.014714,0.689509,1
4,-0.360507,-0.376285,0.566360,0.096222,-0.127541,-0.108158,1.195046,-0.013153,-0.027859,0.092192,...,0.642650,-0.034528,-0.536317,0.908623,-0.090958,0.127925,-0.287978,-0.014714,0.689509,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138042,-0.360507,-0.376285,0.313716,0.047541,-0.127541,-0.108158,0.415868,-0.014266,-0.027859,0.092192,...,-0.386936,-0.246188,-0.174185,-0.943772,1.130480,-0.059242,0.247350,-0.014714,-1.881652,0
138043,-0.360507,2.357301,-0.418468,-0.037697,-0.127541,-0.108158,-0.300756,-0.014266,-0.027859,-2.050128,...,-0.386936,-0.246188,0.547197,-0.248913,-0.167976,-0.039635,0.192239,-0.027153,0.465486,0
138044,-0.360507,-0.376285,0.003432,0.335871,-0.127541,-0.108158,0.096817,-0.014266,-0.027859,0.092192,...,-0.386936,-0.246188,0.427047,0.275771,1.269346,-0.037439,-0.042635,-0.014714,0.385899,0
138045,-0.360507,2.357301,-0.341995,-0.640440,-0.127541,-0.108158,-0.247591,-0.014266,-0.027859,-2.050128,...,-0.386936,-0.246188,-0.022301,-0.220938,-0.341944,-0.059242,-0.264354,-0.027153,-1.881652,0


In [16]:
def traintest_split(df):
    X = df.drop(columns=['legitimate'])
    y = df['legitimate']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    return X_train, X_test, y_train, y_test

In [17]:
def predict(test_data):
    with open (f'model.pkl', 'rb') as f:
        model = pickle.load(f)

    y_pred = model.predict(test_data)
    return y_pred

In [18]:
def train_random_forest(X_train, X_test, y_train, y_test):
    model_type = "Random Forest"
    print(model_type, "classifier:") 
    model = RandomForestClassifier()
    start_time_train = time.time()  # Start time

    model.fit(X_train, y_train)  # Fit the classifier
        
    end_time_train = time.time()  # End time
    time_taken_train = end_time_train - start_time_train  # Time taken to run the code

    print(f"Time taken to train the {model_type} model: {time_taken_train} seconds")
        
    # Make predictions
    start_test = time.time()
    y_pred = model.predict(X_test)
    end_test = time.time()
    time_taken_test = end_test - start_test  # Time taken to run the code

    print(f"Time taken to test the {model_type} model: {time_taken_test} seconds")

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    print("Accuracy:", acc)
    print("Precision:", prec)
    print("Recall:", recall)
    print("F1 score:", f1)

    with open(f'model.pkl', 'wb') as f:
        pickle.dump(model, f)

In [ ]:
def train_ann(X_train, X_test, y_train, y_test):
    input_shape = [X_train.shape[1]]
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(units=64, activation='relu', input_shape=input_shape),
        tf.keras.layers.Dense(units=64, activation='relu'),
        tf.keras.layers.Dense(units=1)
    ])

    model.build()

    print(model.summary())
    start_time_train = time.time()  # Start time

    model.compile(optimizer='adam', loss='mae', metrics=['accuracy'])  
    earlystopping = callbacks.EarlyStopping(monitor="val_loss",
                                                mode="min",
                                                patience=5,
                                                restore_best_weights=True)

    history = model.fit(X_train, y_train, validation_data=(X_test,y_test), batch_size=256, epochs=60,callbacks=[earlystopping])
            
    end_time_train = time.time()  # End time
    time_taken_train = end_time_train - start_time_train  # Time taken to run the code
    with open(f'model.pkl', 'wb') as f:
        pickle.dump(model, f)


In [ ]:
def train(file_path):
    df = pd.read_csv(file_path)
    df = preprocess(df)
    X_train, X_test, y_train, y_test = traintest_split(df)

In [ ]:
def predict(test_data):
    with open (f'model.pkl', 'rb') as f:
        model = pickle.load(f)

    y_pred = model.predict(test_data)
    return y_pred

In [ ]:
def cross_validate(model, X, y):
    start_time_cv = time.time() 
    
    cv_scores = cross_val_score(model, X, y, cv=5)
    
    end_time_cv = time.time()  # End time
    time_taken_cv = end_time_cv - start_time_cv  # Time taken to run the code

    print(f"Time taken to cross-validate the model: {time_taken_cv} seconds")
    print("Cross-validation scores:", cv_scores)
    print("Mean CV accuracy:", cv_scores.mean())
    print("Standard deviation of CV accuracy:", cv_scores.std())